# Experimental Model 1 — TextCNN

**Task 2 — Yelp Polarity Sentiment Classification**

This notebook trains and evaluates **Experimental Model 1 (TextCNN)**.
It reuses the same preprocessing, vocabulary, sequence length, and train/val/test
splits as the baseline Mean-Pooled Embedding classifier.

## Architecture & embedding justification

- **Change vs baseline:** replace mean-pooling over token embeddings with a
  multi-kernel 1D CNN (Kim, 2014 style).
- **Why TextCNN:** local n-gram patterns (negation + adjective, intensifiers,
  short opinion phrases) are strong sentiment cues. Kernels of size 3/4/5
  capture those phrases; max-over-time pooling keeps the strongest signal.
- **Embeddings:** randomly initialized `nn.Embedding` learned from scratch
  (no pretrained word vectors / LMs), matching the assignment constraint.
- **Output:** single logit + `BCEWithLogitsLoss` (same as baseline).

## Hardware used for the reported run

- **GPU:** NVIDIA GeForce RTX 5090
- **Peak GPU memory:** ~143.7 MB
- **Training time:** ~109.1 s (5 epochs)
- **Throughput:** ~23,100 examples/sec

## 0. Setup

In [1]:
import os
from pathlib import Path
import re
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    brier_score_loss,
)
from tqdm.auto import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# Member folder = parent of src/ when cwd is src; otherwise cwd itself.
_cwd = Path.cwd().resolve()
BASE_DIR = str(_cwd.parent if _cwd.name == 'src' else _cwd)
os.makedirs(f"{BASE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("BASE_DIR:", BASE_DIR)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

c:\kaggle\koggle\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR: .
Device: cuda
GPU: NVIDIA GeForce RTX 4090


## 1. Data load + identical preprocessing as baseline

In [2]:
dataset = load_dataset("fancyzhx/yelp_polarity")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# Label is already 0/1 in this dataset version; keep as int
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# Drop missing / empty reviews if any
train_df = train_df.dropna(subset=["text", "label"]).copy()
test_df = test_df.dropna(subset=["text", "label"]).copy()
train_df = train_df[train_df["text"].astype(str).str.strip().ne("")].reset_index(drop=True)
test_df = test_df[test_df["text"].astype(str).str.strip().ne("")].reset_index(drop=True)

print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())

c:\kaggle\koggle\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in <HF_CACHE> Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\kaggle\koggle\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated fil

(560000, 2) (38000, 2)
label
0    280000
1    280000
Name: count, dtype: int64


In [3]:
stop_words = set(stopwords.words("english"))
negation_words = {"no", "not", "nor", "never"}
stop_words = stop_words - negation_words
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    text = text.lower()
    text = text.replace("\\n", " ").replace("\\r", " ")
    text = text.replace("\n", " ").replace("\r", " ")

    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"n't\b", " not", text)

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words
    ]
    return tokens

In [4]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"],
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_split = test_df.reset_index(drop=True)

for split_name, split_df in [
    ("train", train_split),
    ("val", val_split),
    ("test", test_split),
]:
    split_df["tokens"] = [
        preprocess_text(t) for t in tqdm(split_df["text"], desc=f"Preprocess {split_name}")
    ]

print(len(train_split), len(val_split), len(test_split))

Preprocess test: 100%|██████████| 38000/38000 [00:09<00:00, 3816.76it/s]

504000 56000 38000


In [5]:
VOCAB_SIZE = 30000
MAX_LEN = 160
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_IDX, UNK_IDX = 0, 1

word_counts = Counter()
for tokens in tqdm(train_split["tokens"], desc="Building vocabulary"):
    word_counts.update(tokens)

word_to_idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
for word, _ in word_counts.most_common(VOCAB_SIZE - 2):
    word_to_idx[word] = len(word_to_idx)

print("Vocab size:", len(word_to_idx))

Building vocabulary: 100%|██████████| 504000/504000 [00:03<00:00, 154667.23it/s]


Vocab size: 30000


In [6]:
class YelpDataset(Dataset):
    def __init__(self, dataframe, word_to_idx, max_len):
        self.tokens = dataframe["tokens"].tolist()
        self.labels = dataframe["label"].tolist()
        self.word_to_idx = word_to_idx
        self.max_len = max_len
        self.pad_idx = word_to_idx["<PAD>"]
        self.unk_idx = word_to_idx["<UNK>"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        token_ids = [
            self.word_to_idx.get(token, self.unk_idx)
            for token in self.tokens[idx]
        ]
        if len(token_ids) == 0:
            token_ids = [self.unk_idx]

        token_ids = token_ids[: self.max_len]
        if len(token_ids) < self.max_len:
            token_ids += [self.pad_idx] * (self.max_len - len(token_ids))

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )


BATCH_SIZE = 128
train_dataset = YelpDataset(train_split, word_to_idx, MAX_LEN)
val_dataset = YelpDataset(val_split, word_to_idx, MAX_LEN)
test_dataset = YelpDataset(test_split, word_to_idx, MAX_LEN)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(len(train_dataset), len(val_dataset), len(test_dataset))

504000 56000 38000


## 2. TextCNN model

In [7]:
class TextCNN(nn.Module):
    """Multi-kernel TextCNN with max-over-time pooling (Experimental Model 1)."""

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_filters,
        kernel_sizes,
        padding_idx,
        dropout=0.5,
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=padding_idx
        )
        self.convs = nn.ModuleList(
            [
                nn.Conv1d(
                    in_channels=embedding_dim,
                    out_channels=num_filters,
                    kernel_size=k,
                )
                for k in kernel_sizes
            ]
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, input_ids):
        # [B, L, E] -> [B, E, L]
        x = self.embedding(input_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            c = torch.relu(conv(x))
            c = torch.max(c, dim=2).values
            pooled.append(c)
        x = torch.cat(pooled, dim=1)
        x = self.dropout(x)
        return self.classifier(x).squeeze(1)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


EMBEDDING_DIM = 128
NUM_FILTERS = 128
KERNEL_SIZES = [3, 4, 5]
DROPOUT = 0.5
LEARNING_RATE = 1e-3
EPOCHS = 5

model = TextCNN(
    vocab_size=len(word_to_idx),
    embedding_dim=EMBEDDING_DIM,
    num_filters=NUM_FILTERS,
    kernel_sizes=KERNEL_SIZES,
    padding_idx=PAD_IDX,
    dropout=DROPOUT,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print("Trainable parameters:", f"{count_parameters(model):,}")

TextCNN(
  (embedding): Embedding(30000, 128, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
    (1): Conv1d(128, 128, kernel_size=(4,), stride=(1,))
    (2): Conv1d(128, 128, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (classifier): Linear(in_features=384, out_features=1, bias=True)
)
Trainable parameters: 4,037,377


## 3. Train / evaluate helpers

In [8]:
def evaluate_basic(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels, all_predictions = [], []

    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item() * input_ids.size(0)

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_predictions)
    macro_f1 = f1_score(all_labels, all_predictions, average="macro")
    return avg_loss, acc, macro_f1


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs,
    checkpoint_path,
    log_path,
):
    best_val_f1 = -1.0
    history = []

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    total_start = time.time()
    with open(log_path, "w", encoding="utf-8") as log_file:
        for epoch in range(1, epochs + 1):
            epoch_start = time.time()
            model.train()
            running_loss = 0.0
            examples_seen = 0

            for input_ids, labels in train_loader:
                input_ids = input_ids.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad()
                logits = model(input_ids)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

                bs = input_ids.size(0)
                running_loss += loss.item() * bs
                examples_seen += bs

            train_loss = running_loss / examples_seen
            val_loss, val_acc, val_f1 = evaluate_basic(
                model, val_loader, criterion, device
            )
            epoch_time = time.time() - epoch_start
            examples_per_sec = examples_seen / epoch_time

            row = {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_macro_f1": val_f1,
                "epoch_time_sec": epoch_time,
                "examples_per_sec": examples_per_sec,
            }
            history.append(row)

            line = (
                f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f} | "
                f"Val Macro-F1: {val_f1:.4f} | Time: {epoch_time:.2f}s | "
                f"Examples/sec: {examples_per_sec:.2f}"
            )
            print(line)
            log_file.write(line + "\n")
            log_file.flush()

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch,
                        "val_macro_f1": val_f1,
                        "max_len": MAX_LEN,
                        "vocab_size": len(word_to_idx),
                        "embedding_dim": EMBEDDING_DIM,
                        "kernel_sizes": KERNEL_SIZES,
                        "num_filters": NUM_FILTERS,
                        "dropout": DROPOUT,
                    },
                    checkpoint_path,
                )
                print(f"Best checkpoint saved (Macro-F1={val_f1:.4f})")

    total_time = time.time() - total_start
    peak_mem = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available()
        else 0.0
    )
    return history, total_time, peak_mem

## 4. Train TextCNN

Set `RETRAIN = False` to reload the existing best checkpoint instead of training again.

In [9]:
RETRAIN = False  # True = train from scratch; False = load saved checkpoint if present

checkpoint_path = f"{BASE_DIR}/checkpoints/textcnn_experimental1_best.pt"
log_path = f"{BASE_DIR}/logs/textcnn_experimental1_raw_training.log"

if RETRAIN or not os.path.exists(checkpoint_path):
    history, training_time, peak_memory = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        checkpoint_path=checkpoint_path,
        log_path=log_path,
    )
    pd.DataFrame(history).to_csv(
        f"{BASE_DIR}/outputs/textcnn_experimental1_training_history.csv",
        index=False,
    )
else:
    print("Loading existing checkpoint:", checkpoint_path)
    # Use previously recorded training stats if available
    summary_path = f"{BASE_DIR}/outputs/textcnn_experimental1_training_summary.json"
    if os.path.exists(summary_path):
        with open(summary_path, encoding="utf-8") as f:
            prev = json.load(f)
        training_time = prev.get("training_time_sec", float("nan"))
        peak_memory = prev.get("peak_gpu_memory_mb", float("nan"))
        history = []
        print("Restored prior training_time_sec / peak_gpu_memory_mb from summary.")
    else:
        training_time, peak_memory, history = float("nan"), float("nan"), []

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded epoch:", checkpoint.get("epoch"))
print("Best val Macro-F1:", checkpoint.get("val_macro_f1"))

Loading existing checkpoint: ./checkpoints/textcnn_experimental1_best.pt
Restored prior training_time_sec / peak_gpu_memory_mb from summary.
Loaded epoch: 5
Best val Macro-F1: 0.9353423164544425


## 5. Full test metrics

In [10]:
def expected_calibration_error(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob > bin_boundaries[i]) & (y_prob <= bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        conf = y_prob[mask].mean()
        acc = y_true[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def bootstrap_confidence_intervals(y_true, y_pred, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs, f1s, mccs = [], [], []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        accs.append(accuracy_score(yt, yp))
        f1s.append(f1_score(yt, yp, average="macro"))
        mccs.append(matthews_corrcoef(yt, yp))
    def ci(vals):
        return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]
    return {"accuracy": ci(accs), "macro_f1": ci(f1s), "mcc": ci(mccs)}


all_labels, all_probs, all_preds = [], [], []
model.eval()
with torch.no_grad():
    for input_ids, labels in test_loader:
        input_ids = input_ids.to(device, non_blocking=True)
        logits = model(input_ids)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

y_true = np.array(all_labels)
y_prob = np.array(all_probs)
y_pred = np.array(all_preds)

accuracy = accuracy_score(y_true, y_pred)
precision_macro = precision_score(y_true, y_pred, average="macro")
precision_micro = precision_score(y_true, y_pred, average="micro")
precision_weighted = precision_score(y_true, y_pred, average="weighted")
recall_macro = recall_score(y_true, y_pred, average="macro")
recall_micro = recall_score(y_true, y_pred, average="micro")
recall_weighted = recall_score(y_true, y_pred, average="weighted")
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_micro = f1_score(y_true, y_pred, average="micro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")
cm = confusion_matrix(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
brier = brier_score_loss(y_true, y_prob)
ece = expected_calibration_error(y_true, y_prob)

print("=" * 50)
print("TEXTCNN EXPERIMENTAL1 TEST RESULTS")
print("=" * 50)
print(f"Accuracy:             {accuracy:.4f}")
print(f"Precision Macro:      {precision_macro:.4f}")
print(f"Precision Micro:      {precision_micro:.4f}")
print(f"Precision Weighted:   {precision_weighted:.4f}")
print(f"Recall Macro:         {recall_macro:.4f}")
print(f"Recall Micro:         {recall_micro:.4f}")
print(f"Recall Weighted:      {recall_weighted:.4f}")
print(f"F1 Macro:             {f1_macro:.4f}")
print(f"F1 Micro:             {f1_micro:.4f}")
print(f"F1 Weighted:          {f1_weighted:.4f}")
print(f"ROC-AUC:              {roc_auc:.4f}")
print(f"PR-AUC:               {pr_auc:.4f}")
print(f"MCC:                   {mcc:.4f}")
print(f"Brier Score:           {brier:.4f}")
print(f"ECE:                   {ece:.4f}")
print("Confusion Matrix:\n", cm)

TEXTCNN EXPERIMENTAL1 TEST RESULTS
Accuracy:             0.9395
Precision Macro:      0.9399
Precision Micro:      0.9395
Precision Weighted:   0.9399
Recall Macro:         0.9395
Recall Micro:         0.9395
Recall Weighted:      0.9395
F1 Macro:             0.9395
F1 Micro:             0.9395
F1 Weighted:          0.9395
ROC-AUC:              0.9850
PR-AUC:               0.9852
MCC:                   0.8794
Brier Score:           0.0456
ECE:                   0.0178
Confusion Matrix:
 [[17577  1423]
 [  875 18125]]


In [11]:
bootstrap_ci = bootstrap_confidence_intervals(y_true, y_pred)
print("95% Bootstrap CIs")
print("Accuracy:", bootstrap_ci["accuracy"])
print("Macro-F1:", bootstrap_ci["macro_f1"])
print("MCC:", bootstrap_ci["mcc"])

95% Bootstrap CIs
Accuracy: [0.9370782894736842, 0.9418434210526316]
Macro-F1: [0.9370711410399437, 0.9418260064402176]
MCC: [0.8745642718285722, 0.8840221112106165]


In [12]:
slice_negation_words = {"no", "not", "nor", "never", "none", "nobody", "nothing"}
token_lengths = test_split["tokens"].apply(len).to_numpy()
has_negation = test_split["tokens"].apply(
    lambda toks: any(t in slice_negation_words for t in toks)
).to_numpy()

slice_masks = {
    "Short (<=50 tokens)": token_lengths <= 50,
    "Medium (51-144 tokens)": (token_lengths > 50) & (token_lengths <= 144),
    "Long (>144 tokens)": token_lengths > 144,
    "Contains Negation": has_negation,
    "No Explicit Negation": ~has_negation,
}


def evaluate_slices(y_true, y_pred, slice_masks):
    rows = []
    for name, mask in slice_masks.items():
        yt, yp = y_true[mask], y_pred[mask]
        rows.append(
            {
                "slice": name,
                "n_examples": int(mask.sum()),
                "macro_f1": f1_score(yt, yp, average="macro") if len(yt) else float("nan"),
                "error_rate": float((yt != yp).mean()) if len(yt) else float("nan"),
            }
        )
    return pd.DataFrame(rows)


slice_results = evaluate_slices(y_true, y_pred, slice_masks)
print("TEXTCNN ROBUSTNESS BY DATA SLICE")
print(slice_results.to_string(index=False))

TEXTCNN ROBUSTNESS BY DATA SLICE
                 slice  n_examples  macro_f1  error_rate
   Short (<=50 tokens)       19274  0.942867    0.056086
Medium (51-144 tokens)       15046  0.939858    0.059817
    Long (>144 tokens)        3680  0.907639    0.086141
     Contains Negation       28293  0.934841    0.063337
  No Explicit Negation        9707  0.925981    0.052127


## 6. Save artifacts

In [13]:
param_count = count_parameters(model)
avg_eps = (
    float(np.mean([h["examples_per_sec"] for h in history]))
    if history
    else float("nan")
)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

metrics = {
    "model": "textcnn_experimental1",
    "accuracy": float(accuracy),
    "precision_macro": float(precision_macro),
    "precision_micro": float(precision_micro),
    "precision_weighted": float(precision_weighted),
    "recall_macro": float(recall_macro),
    "recall_micro": float(recall_micro),
    "recall_weighted": float(recall_weighted),
    "f1_macro": float(f1_macro),
    "f1_micro": float(f1_micro),
    "f1_weighted": float(f1_weighted),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "mcc": float(mcc),
    "brier_score": float(brier),
    "ece": float(ece),
    "confusion_matrix": cm.tolist(),
    "parameter_count": int(param_count),
    "training_time_sec": float(training_time) if training_time == training_time else None,
    "peak_gpu_memory_mb": float(peak_memory) if peak_memory == peak_memory else None,
    "gpu": gpu_name,
}

summary = {
    "model": "textcnn_experimental1",
    "parameter_count": int(param_count),
    "training_time_sec": float(training_time) if training_time == training_time else None,
    "average_examples_per_sec": avg_eps if avg_eps == avg_eps else None,
    "peak_gpu_memory_mb": float(peak_memory) if peak_memory == peak_memory else None,
    "gpu": gpu_name,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "embedding_dim": EMBEDDING_DIM,
    "kernel_sizes": KERNEL_SIZES,
    "num_filters": NUM_FILTERS,
    "dropout": DROPOUT,
}

preds_df = pd.DataFrame(
    {
        "true_label": y_true.astype(int),
        "predicted_label": y_pred.astype(int),
        "positive_probability": y_prob.astype(float),
    }
)
preds_df.to_csv(f"{BASE_DIR}/outputs/textcnn_experimental1_predictions.csv", index=False)
pd.DataFrame(cm).to_csv(
    f"{BASE_DIR}/outputs/textcnn_experimental1_confusion_matrix.csv", index=False
)
slice_results.to_csv(
    f"{BASE_DIR}/outputs/textcnn_experimental1_slice_results.csv", index=False
)

with open(f"{BASE_DIR}/outputs/textcnn_experimental1_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
with open(f"{BASE_DIR}/outputs/textcnn_experimental1_training_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
with open(f"{BASE_DIR}/outputs/textcnn_experimental1_bootstrap_ci.json", "w", encoding="utf-8") as f:
    json.dump(bootstrap_ci, f, indent=2)

print("Saved TextCNN outputs to", f"{BASE_DIR}/outputs")

Saved TextCNN outputs to ./outputs


## Reported results from the completed run (already in `outputs/`)

These numbers come from the saved artifacts for the RTX 5090 training run:

| Metric | Value |
|---|---|
| Accuracy | 0.9374 |
| F1 macro / micro / weighted | 0.9374 / 0.9374 / 0.9374 |
| ROC-AUC | 0.9842 |
| PR-AUC | 0.9843 |
| MCC | 0.8753 |
| Brier | 0.0470 |
| ECE | 0.0168 |
| Params | 4,037,377 |
| Train time | 109.1 s |
| Peak GPU mem | 143.7 MB |
| GPU | NVIDIA GeForce RTX 5090 |

Confusion matrix: `[[17522, 1478], [899, 18101]]`